# Day 16 — 샘플 표와 유전자 행렬을 안전하게 연결하기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-3/day-16.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-3/day-16.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 순서로 붙이면 조용히 틀린다

In [ ]:
import pandas as pd

expr = pd.DataFrame({"sample_id": ["S3", "S1", "S2"], "GeneX": [10, 200, 220]})
meta = pd.DataFrame({"sample_id": ["S1", "S2", "S3"], "group": ["FLT", "FLT", "GC"]})

wrong = expr.copy()
wrong["group"] = meta["group"].values           # 순서만 믿고 붙임
right = expr.merge(meta, on="sample_id")        # ID로 이음

print(wrong.groupby("group")["GeneX"].mean().to_string())
print(right.groupby("group")["GeneX"].mean().to_string())

### 예시 2 — 한쪽에만 있는 샘플 찾기

In [ ]:
meta = pd.DataFrame({"sample_id": ["S1", "S2", "S3", "S4"],
                     "group": ["FLT", "FLT", "GC", "GC"]})
files = pd.DataFrame({"sample_id": ["S1", "S2", "S3 ", "S5"],     # "S3 " 뒤에 공백
                      "file": ["a.gz", "b.gz", "c.gz", "e.gz"]})

check = meta.merge(files, on="sample_id", how="outer", indicator=True)
print(check.to_string(index=False))

### 예시 3 — 중복이 있으면 `validate`가 멈춰 준다

In [ ]:
meta_dup = pd.DataFrame({"sample_id": ["S1", "S1", "S2"], "group": ["FLT", "GC", "FLT"]})
expr = pd.DataFrame({"sample_id": ["S1", "S2"], "GeneX": [200, 220]})

print("중복된 ID:", meta_dup.loc[meta_dup["sample_id"].duplicated(keep=False), "sample_id"].unique())
try:
    expr.merge(meta_dup, on="sample_id", validate="one_to_one")
except pd.errors.MergeError as e:
    print("멈춤:", e)

## 직접 해보기

1. 노트북 `02_osdr_qc_and_pca.ipynb`에 "Day 16" 제목 셀을 만들고 설정 셀을 실행합니다.

In [ ]:
# 이 셀은 Day 15의 산출물 파일이 필요합니다. Colab은 노트북마다 런타임이 따로라서,
# 그 파일을 이 런타임에 업로드하거나 Google Drive에 두고 경로를 맞추세요.
from pathlib import Path
import gzip
import io
import pandas as pd

MODE = "practice"          # 연습이 끝나면 "real"로 바꾸세요
RAW_DIR = Path("../data/osdr_536_practice" if MODE == "practice" else "../data/osdr_536")
assert RAW_DIR.exists(), f"{RAW_DIR} 폴더가 없습니다. Day 15를 먼저 실행하세요."
OUT_DIR = RAW_DIR / "processed"
OUT_DIR.mkdir(exist_ok=True)

2. 샘플 표와 assay 표를 읽고, 열 이름을 **먼저 눈으로 확인**합니다. 실제 샘플 표에는 열이 70개가 넘으므로 우리가 쓸 세 개가 정말 있는지 확인하는 것입니다.

In [ ]:
s_table = pd.read_csv(next(RAW_DIR.glob("s_*.txt")), sep="\t")
a_table = pd.read_csv(next(RAW_DIR.glob("a_*.txt")), sep="\t")

need_s = ["Sample Name", "Source Name", "Factor Value[Spaceflight]"]
need_a = ["Sample Name", "Array Data File"]
print("샘플 표 크기:", s_table.shape, "| assay 표 크기:", a_table.shape)
print("필요한 열이 모두 있는가:", set(need_s) <= set(s_table.columns), set(need_a) <= set(a_table.columns))

3. 두 표를 `Sample Name`으로 잇고, 읽기 쉬운 열 이름과 짧은 군 이름(`FLT`, `GC`)을 붙입니다.

In [ ]:
samples = s_table[need_s].merge(a_table[need_a], on="Sample Name",
                                how="outer", validate="one_to_one", indicator=True)
samples.columns = ["sample_id", "source", "condition", "file", "_merge"]
samples["group"] = samples["condition"].map({"Space Flight": "FLT", "Ground Control": "GC"})
print(samples.head(3).to_string(index=False))

4. RCC 파일 하나에서 `<Code_Summary>` 표만 꺼내는 함수를 만들고, assay 표가 알려 준 파일 이름대로 모두 읽어 세로로 쌓습니다.

In [ ]:
def read_rcc(path):
    """RCC(.gz) 파일 하나에서 Code_Summary 표(CodeClass, Name, Accession, Count)를 꺼낸다."""
    with gzip.open(path, "rt") as f:
        text = f.read()
    block = text.split("<Code_Summary>")[1].split("</Code_Summary>")[0]
    return pd.read_csv(io.StringIO(block.strip()))

long = pd.concat(
    [read_rcc(RAW_DIR / row.file).assign(sample_id=row.sample_id) for row in samples.itertuples()],
    ignore_index=True,
)
print(long.shape)
print(long.head(3).to_string(index=False))

5. 세로로 긴 표를 "샘플(행) × 유전자(열)" 행렬로 바꿉니다. 같은 샘플에 같은 이름이 두 번 있으면 `pivot`이 오류를 내므로, 이 줄이 통과했다는 것 자체가 중복 검사입니다.

In [ ]:
counts = long.pivot(index="sample_id", columns="Name", values="Count")
probes = long.drop_duplicates("Name")[["Name", "CodeClass", "Accession"]].reset_index(drop=True)
print("행렬 크기(샘플, 프로브):", counts.shape)
print(probes["CodeClass"].value_counts().to_string())

6. 네 가지 확인을 숫자로 남기고 QC 요약표로 저장합니다.

In [ ]:
on_disk = {p.name for p in RAW_DIR.glob("*.RCC.gz")}
in_meta = set(samples["file"].dropna())

qc = pd.DataFrame([
    ("샘플 표의 샘플 수", len(s_table)),
    ("폴더의 RCC 파일 수", len(on_disk)),
    ("메타데이터에만 있는 파일", len(in_meta - on_disk)),
    ("폴더에만 있는 파일", len(on_disk - in_meta)),
    ("양쪽 표에 모두 있는 샘플", int((samples["_merge"] == "both").sum())),
    ("중복된 sample_id", int(samples["sample_id"].duplicated().sum())),
    ("군 정보가 없는 샘플", int(samples["group"].isna().sum())),
    ("FLT 샘플 수", int((samples["group"] == "FLT").sum())),
    ("GC 샘플 수", int((samples["group"] == "GC").sum())),
    ("샘플마다 프로브 수가 같은가", long.groupby("sample_id")["Name"].nunique().nunique() == 1),
    ("행렬의 빈 칸 수", int(counts.isna().sum().sum())),
    ("Endogenous 유전자 수", int((probes["CodeClass"] == "Endogenous").sum())),
], columns=["확인 항목", "값"])
print(qc.to_string(index=False))

counts.to_csv(OUT_DIR / "counts_raw.csv")
probes.to_csv(OUT_DIR / "probes.csv", index=False)
samples.drop(columns="_merge").to_csv(OUT_DIR / "samples.csv", index=False)
qc.to_csv(OUT_DIR / "qc_summary.csv", index=False)